# Asteroid Diameter Prediction with Linear Regression (SGD)

**Project Goal:** To develop a Linear Regression model to predict asteroid diameter based on orbital and physical features. The Stochastic Gradient Descent (SGD) optimizer will be
implemented from scratch.

## Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Data Loading and Preparation

### Load Data

In [3]:
df = pd.read_csv('NASA_JPL_Dataset.csv')

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe().T

### Preprocessing

#### Feature Engineering

In [4]:
# Add new features. (e.g. if we have feats like x, y and z, you can add x^i times y^j times z^k where at i + j + k >= 2)
# Warning: it can overfit the model.
df['log_diameter'] = np.log(df['diameter'])

#### Outlier Detection

In [ ]:
df['rms'].describe()

In [ ]:
threshold = 1
high_rms = df[df['rms'] > threshold]
print(f"Number of rows with RMS > {threshold}: {len(high_rms)}")
print(f"Percentage: {len(high_rms) / len(df) * 100:.2f}%")

In [ ]:
# Drop some outlier instances based on std and percentile.
df = df[df['rms'] <= 1]

#### Split Data

In [5]:
X = df.drop(columns=['log_diameter','diameter'])  # Features
y = df['diameter']
y_log = df['log_diameter']  # Target variable
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Remove irrelevant columns

In [ ]:
# # define list of irrelevant columns
cols_to_drop = ['full_name', 'class', 'om', 'ma', 'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w',
                'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_per', 'diameter_sigma', 'rms']

X_train.drop(columns=cols_to_drop, inplace=True)
X_test.drop(columns=cols_to_drop, inplace=True)

#### EDA and Visualization

In [ ]:
# Example: Correlation matrix
numeric_columns = df.select_dtypes(include=['int64', 'float64'])
plt.figure(figsize=(20, 6))
Corr = numeric_columns.corr()
sns.heatmap(Corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap for Numerical Features')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['diameter'], kde=True)
plt.title('Distribution of Diameter')
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(df['H'], kde=True)
plt.title('Distribution of Absolute Magnitude (H)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='H', y='diameter', data=df)
plt.title('Diameter vs Absolute Magnitude')
plt.xlabel('Absolute Magnitude (H)')
plt.ylabel('Diameter (km)')
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(x='H', y='log_diameter', data=df)
plt.title('Log(Diameter) vs Absolute Magnitude (H) [After Transformation]')
plt.xlabel('Absolute Magnitude (H)')
plt.ylabel('Log(Diameter)')
plt.show()

#### Encoding

In [ ]:
categorical = X_train.select_dtypes(include='object').columns
label_encoder = {}
for col in categorical:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    label_encoder[col] = le

#### Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Model Implementation and Training (From Scratch)

In [ ]:
class SGDRegressorScratch:
    """
    Implementation of the SGD Regression model from scratch.

    Parameters:
    -----------
    learning_rate (float): The learning rate for weight updates.
    n_epochs (int): The number of passes over the entire dataset.
    random_state (int): Ensures reproducible results for data shuffling.
    """

    def __init__(self, learning_rate, n_epochs, random_state, batch_size, beta=0.9,
                 lr_decay=0.0):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.batch_size = batch_size
        self.beta = beta
        self.lr_decay = lr_decay
        self.weights = None
        self.loss_history = []

    def augment_x(self, X_train):
        """
        Adds a column of ones to X_train to incorporate the bias (intercept) term
        in the weight vector.

        """
        return np.insert(X_train, X_train.shape[1], 1, axis=1)

    def fit(self, X, y):
        """
        Fit the model to the training data.

        Parameters:
        -----------
        X (array-like): Feature matrix of shape (n_samples, n_features).
        y (array-like): Target vector of shape (n_samples,).
        """
        x_train = np.asarray(X)
        y_train = np.asarray(y)
        x_batch = None
        y_batch = None
        Xaug_train = self.augment_x(x_train)
        n_samples, n_features_Xaug = Xaug_train.shape
        np.random.seed(self.random_state)

        # initialize the primary wights vector
        self.weights = np.random.randn(n_features_Xaug) * 0.01
        velocity = np.zeros_like(self.weights)
        for epoch in range(self.n_epochs):
            shuffled_indices = np.random.permutation(n_samples)

            lr = self.learning_rate / (1 + self.lr_decay * epoch)
            for start_idx in range(0, n_samples, self.batch_size):
                end_idx = min(start_idx + self.batch_size, n_samples)
                batch_indices = shuffled_indices[start_idx:end_idx]
                x_batch = Xaug_train[batch_indices]
                y_batch = y_train[batch_indices]
                n_batch = x_batch.shape[0]

                y_pred_batch = x_batch @ self.weights
                error_batch = y_pred_batch - y_batch
                gradient_batch = (2 * x_batch.T @ error_batch) / n_batch

                velocity = self.beta * velocity + (1 - self.beta) * gradient_batch
                self.weights -= lr * velocity
                #self.weights -= self.learning_rate * gradient_batch

            # add mse to loss_history
            y_pred_full = Xaug_train @ self.weights
            mse = np.mean((y_train - y_pred_full) ** 2)
            self.loss_history.append(mse)
            print(f"Epoch {epoch + 1}/{self.n_epochs} - MSE: {mse:.6f}")
        return self

    def plot_loss_curve(self):
        plt.plot(range(1, len(self.loss_history) + 1), self.loss_history, marker='.')
        plt.xlabel('Epoch')
        plt.ylabel('Mean Squared Error (MSE)')
        plt.title('Loss Curve During Training')
        plt.grid(True)
        plt.show()

    def predict(self, X):
        """
        Predict values for new data.

        Parameters:
        -----------
        X (array-like): Feature matrix of shape (n_samples, n_features).

        Returns:
        -------
        array: Predicted values.
        """

        if self.weights is None:
            raise RuntimeError("call fit(), model should be fitted before predicting")

        X_aug = self.augment_x(np.asarray(X))
        y_pred = X_aug @ self.weights
        return y_pred

In [ ]:
Epochs = 200
Learning_rate = 0.005
random_state = 42
batch_size = 2000  #tedadesh bishtar she behtareh
sgd_scratch_model = SGDRegressorScratch(
    n_epochs=Epochs,
    learning_rate=Learning_rate,
    random_state=random_state,
    batch_size=batch_size,
    beta=0.9,
    lr_decay=0.001
)

sgd_scratch_model.fit(X_train, y_train.values)
sgd_scratch_model.plot_loss_curve()

y_pred_scratch = sgd_scratch_model.predict(X_test)

R2_Score = r2_score(y_test, y_pred_scratch)
MAE_Score = mean_absolute_error(y_test, y_pred_scratch)
MSE_Score = mean_squared_error(y_test, y_pred_scratch)

print(f"R² Scratch SGD Score: {R2_Score:.4f}")
print(f"MAE SGD Score: {MAE_Score:.4f}")
print(f"MSE SGD Score: {MSE_Score:.4f}")